In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

In [ ]:
def get_gasreg_data(gasreg):
    data = pd.concat([
        gasreg_log_mult_diffs[[gasreg]].add_suffix('_price'),
        gasreg_degree_days.filter(like=gasreg)
    ], axis=1)
 
    return data

def run_ols_regression(gasreg, data):
    data['month'] = data.index.get_level_values('month')
    month_dummies = (
        pd.get_dummies(data['month'], prefix='month')
        .drop(columns='month_5')
        .astype(float)
    )
    
    degree_day_columns = [f'{gasreg}_{dd_var}' for dd_var in ['cdd', 'hdd']]
    temperature_variable = data[degree_day_columns].astype(float)
    
    X = pd.concat([temperature_variable, month_dummies], axis=1)
    X = sm.add_constant(X)
    y = data[f"{gasreg}_price"].values.astype(float)
    
    # The lines below represent an ordinary least-squares regression using a
    # heteroskedasticity- and autocorrelation-consistent (HAC) estimator,
    # which ensures that the standard errors calculated in the regression
    # are robust to heteroskedastic and autocorrelated residuals (both
    # common when working with time series data).
    # The maxlags value represents the maximum number of timesteps
    # (in this case days) across which the estimator adjusts for auto-
    # correlation. The value is calculated using the Stock and Watson rule-of-thumb:
    # number of lags = 0.75 * (number of observations)**(1/3)
    # https://stock.scholars.harvard.edu/sites/g/files/omnuum5911/files/stock/files/aea_2015_lecture4_har_rev.pdf
    maxlags = int(round(0.75 * len(data)**(1/3)))
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})

    return model

def apply_regression_model(gasreg, model, data):
    data['month'] = data.index.get_level_values('month')
    month_dummies = (
        pd.get_dummies(data['month'], prefix='month')
        .drop(columns='month_5')
        .astype(float)
    )
    
    degree_day_columns = [f'{gasreg}_{dd_var}' for dd_var in ['cdd', 'hdd']]
    temperature_variable = data[degree_day_columns].astype(float)

    X_test = pd.concat([temperature_variable, month_dummies], axis=1)
    X_test = sm.add_constant(X_test)
    y_pred = model.predict(X_test)

    return y_pred

In [23]:
# Get daily HDD/CDDs and prices for each gasreg
gasreg_data = pd.read_csv(
    Path('inputs', 'gasreg_regression_data.csv'),
    index_col=['year', 'month', 'day']
)
gasreg_degree_days = (
    gasreg_data[[col for col in gasreg_data.columns if 'dd' in col]]
    .copy()
)
gasreg_prices = (
    gasreg_data[[col for col in gasreg_data.columns if 'price' in col]]
    .copy()
)
gasreg_prices.columns = [col.replace('_price', '') for col in gasreg_prices.columns]

In [24]:
# Calculate average annual prices for each gasreg
# and the daily deviations (log of the multiplicative difference)
# from the annual average price
gasreg_annual_average_prices = (
    gasreg_prices.groupby(gasreg_prices.index.get_level_values('year'))
    .transform('mean')
)
gasreg_log_mult_diffs = np.log(gasreg_prices) - np.log(gasreg_annual_average_prices)

In [25]:
# Fit the regression model using data from 2014-2023 (2024 was used for testing)
train_years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
test_years = [2024]

gasreg_model_params = {}
for gasreg in gasreg_prices.columns.tolist():
    data = get_gasreg_data(gasreg)
    train_data = data.loc[data.index.get_level_values('year').isin(train_years)].copy()
    model = run_ols_regression(gasreg, train_data)

    gasreg_model_params[gasreg] = (
        model.params
        .rename({
            f"{gasreg}_cdd": 'cdd',
            f"{gasreg}_hdd": 'hdd'
        })
    )

In [7]:
# Reformat for ReEDS
regression_params = (
    pd.concat(gasreg_model_params, axis=1)
    .rename({
        'const': 'alpha',
        'cdd': 'beta_CDD',
        'hdd': 'beta_HDD',
        'month_1': 'alpha_JAN',
        'month_2': 'alpha_FEB',
        'month_3': 'alpha_MAR',
        'month_4': 'alpha_APR',
        'month_6': 'alpha_JUN',
        'month_7': 'alpha_JUL',
        'month_8': 'alpha_AUG',
        'month_9': 'alpha_SEP',
        'month_10': 'alpha_OCT',
        'month_11': 'alpha_NOV',
        'month_12': 'alpha_DEC'
    })
)
regression_params.loc['alpha_MAY'] = 0

In [ ]:
# Export
outpath = Path('outputs', 'gasreg_degree_day_price_mult_regression_params.csv')
outpath.parent.mkdir(parents=True, exist_ok=True)
(
    regression_params.loc[[
        'beta_CDD',
        'beta_HDD',
        'alpha',
        'alpha_JAN',
        'alpha_FEB',
        'alpha_MAR',
        'alpha_APR',
        'alpha_MAY',
        'alpha_JUN',
        'alpha_JUL',
        'alpha_AUG',
        'alpha_SEP',
        'alpha_OCT',
        'alpha_NOV',
        'alpha_DEC'
    ]]
    .rename_axis('param')
    .round(3)
    .sort_index(axis=1)
    .to_csv(outpath)
)